In [1]:
try:
    %load_ext autoreload
    %autoreload 2
except Exception:
    pass

import importlib

from pylabrobot.liquid_handling import LiquidHandler
from pylabrobot.liquid_handling.backends import STARBackend
from pylabrobot.resources.hamilton import STARLetDeck, MFX_CAR_L5_base, TIP_CAR_480_A00
from pylabrobot.resources.hamilton.mfx_modules import Hamilton_MFX_plateholder_DWP_metal_tapped
from pylabrobot.resources.diy.grindbio.modules import Hamilton_MFX_plateholder_DWP_metal_tapped_10mm_3dprint
from pylabrobot.resources.alpaqua import Alpaqua_96_magnum_flx
from pylabrobot.resources.bioer.plates import BioER_96_wellplate_Vb_2200uL
from pylabrobot.resources.agenbio.plates import AGenBio_1_troughplate_100000uL_Fl
from pylabrobot.resources import hamilton_96_tiprack_1000uL


In [2]:
# Deck layout for the 96-well APE workflow.
backend = STARBackend()
lh: LiquidHandler = LiquidHandler(backend=backend, deck=STARLetDeck())

deck = STARLetDeck(
  core_grippers="1000uL-at-waste"  # or "1000uL-5mL-on-waste"
) 

tip_car = TIP_CAR_480_A00("tip_car")
lh.deck.assign_child_resource(tip_car, rails=25)
tiprack_1000_1 = hamilton_96_tiprack_1000uL("tips_00")
tiprack_1000_2 = hamilton_96_tiprack_1000uL("tips_01")
tiprack_1000_3 = hamilton_96_tiprack_1000uL("tips_02")
tip_car[0] = tiprack_1000_1
tip_car[1] = tiprack_1000_2
tip_car[2] = tiprack_1000_3
tip_racks = [tiprack_1000_1, tiprack_1000_2, tiprack_1000_3]

# Rails 13: mag plate + wash / waste / elution resources on 10 mm supports.
rail13_modules = {
    0: Hamilton_MFX_plateholder_DWP_metal_tapped_10mm_3dprint("rail13_mag_module"),
    1: Hamilton_MFX_plateholder_DWP_metal_tapped_10mm_3dprint("rail13_waste_module"),
    2: Hamilton_MFX_plateholder_DWP_metal_tapped_10mm_3dprint("rail13_elution_plate_module"),
    3: Hamilton_MFX_plateholder_DWP_metal_tapped_10mm_3dprint("rail13_wash1_module"),
    4: Hamilton_MFX_plateholder_DWP_metal_tapped_10mm_3dprint("rail13_wash2_module"),
}
car_13 = MFX_CAR_L5_base("car_13", modules=rail13_modules)
lh.deck.assign_child_resource(car_13, rails=13)

mag_plate = Alpaqua_96_magnum_flx("mag_plate")
waste_trough = AGenBio_1_troughplate_100000uL_Fl("waste_trough")
elution_plate = BioER_96_wellplate_Vb_2200uL("elution_plate")
wash1_trough = AGenBio_1_troughplate_100000uL_Fl("wash1_trough")
wash2_trough = AGenBio_1_troughplate_100000uL_Fl("wash2_trough")

rail13_modules[0].assign_child_resource(mag_plate)
rail13_modules[1].assign_child_resource(waste_trough)
rail13_modules[2].assign_child_resource(elution_plate)
rail13_modules[3].assign_child_resource(wash1_trough)
rail13_modules[4].assign_child_resource(wash2_trough)

# Rails 19: binding plate + source / flowthrough / binding buffer / elution buffer.
rail19_modules = {
    0: Hamilton_MFX_plateholder_DWP_metal_tapped("rail19_binding_module"),
    1: Hamilton_MFX_plateholder_DWP_metal_tapped("rail19_source_module"),
    2: Hamilton_MFX_plateholder_DWP_metal_tapped("rail19_flowthrough_module"),
    3: Hamilton_MFX_plateholder_DWP_metal_tapped("rail19_binding_trough_module"),
    4: Hamilton_MFX_plateholder_DWP_metal_tapped("rail19_elution_trough_module"),
}
car_19 = MFX_CAR_L5_base("car_19", modules=rail19_modules)
lh.deck.assign_child_resource(car_19, rails=19)

binding_plate = BioER_96_wellplate_Vb_2200uL("binding_plate")
source_plate = BioER_96_wellplate_Vb_2200uL("source_plate")
flowthrough_plate = BioER_96_wellplate_Vb_2200uL("flowthrough_plate")
binding_trough = AGenBio_1_troughplate_100000uL_Fl("binding_trough")
elution_trough = AGenBio_1_troughplate_100000uL_Fl("elution_trough")

rail19_modules[0].assign_child_resource(binding_plate)
rail19_modules[1].assign_child_resource(source_plate)
rail19_modules[2].assign_child_resource(flowthrough_plate)
rail19_modules[3].assign_child_resource(binding_trough)
rail19_modules[4].assign_child_resource(elution_trough)


/tmp/ipykernel_1957077/1145598107.py:27: DeprecationWarning: MFX_CAR_L5_base is deprecated. Use 'hamilton_mfx_carrier_L5_base' instead.
  car_13 = MFX_CAR_L5_base("car_13", modules=rail13_modules)
/tmp/ipykernel_1957077/1145598107.py:30: DeprecationWarning: Alpaqua_96_magnum_flx is deprecated. Use 'alpaqua_96_plateadapter_magnum_flx' instead.
  mag_plate = Alpaqua_96_magnum_flx("mag_plate")
/tmp/ipykernel_1957077/1145598107.py:44: DeprecationWarning: Hamilton_MFX_plateholder_DWP_metal_tapped is deprecated. Use 'hamilton_mfx_plateholder_DWP_metal_tapped' instead.
  0: Hamilton_MFX_plateholder_DWP_metal_tapped("rail19_binding_module"),
/tmp/ipykernel_1957077/1145598107.py:45: DeprecationWarning: Hamilton_MFX_plateholder_DWP_metal_tapped is deprecated. Use 'hamilton_mfx_plateholder_DWP_metal_tapped' instead.
  1: Hamilton_MFX_plateholder_DWP_metal_tapped("rail19_source_module"),
/tmp/ipykernel_1957077/1145598107.py:46: DeprecationWarning: Hamilton_MFX_plateholder_DWP_metal_tapped is depre

In [3]:
await lh.setup(skip_autoload=True)

# STARlet without iSWAP reports 0 arms; keep Co-Re gripper bookkeeping available.
# if lh.backend.num_arms == 0:
#     lh._resource_pickups = {0: None}

# print(lh.deck.get_resource("core_grippers"))


2026-04-22 10:28:18,786 - pylabrobot.io.usb - INFO - Finding USB device...
2026-04-22 10:28:18,795 - pylabrobot.io.usb - INFO - Found USB device.
2026-04-22 10:28:18,798 - pylabrobot.io.usb - INFO - Found endpoints. 
Write:
       ENDPOINT 0x2: Bulk OUT ===============================
       bLength          :    0x7 (7 bytes)
       bDescriptorType  :    0x5 Endpoint
       bEndpointAddress :    0x2 OUT
       bmAttributes     :    0x2 Bulk
       wMaxPacketSize   :   0x40 (64 bytes)
       bInterval        :    0x0 
Read:
       ENDPOINT 0x81: Bulk IN ===============================
       bLength          :    0x7 (7 bytes)
       bDescriptorType  :    0x5 Endpoint
       bEndpointAddress :   0x81 IN
       bmAttributes     :    0x2 Bulk
       wMaxPacketSize   :   0x40 (64 bytes)
       bInterval        :    0x0
2026-04-22 10:28:21,962 - pylabrobot - INFO - Running backend initialization procedure.


STARFirmwareError: {'Pipetting channel 1': UnknownHamiltonError('Unknown command')}, P1VWid0015er30

In [ ]:
# PLR deck setup overview.
# print(lh.summary())
print (lh.backend.request_name_of_last_faulty_parameter())
await lh.backend.request_firmware_version()
print ("iswap_installed:", lh.backend.extended_conf.left_x_drive.iswap_installed)
# print ("core96_installed:", lh.backend.extended_conf.left_x_drive.core96_installed)
print ("core8_installed:", lh.backend.extended_conf.left_x_drive.pip_installed)
print ("num_arms: ", lh.backend.num_arms)

In [ ]:
await lh.pick_up_tips(tiprack_1000_1["A1:H1"], use_channels=[0,1,2,3,4,5,6,7])
# await lh.drop_tips(tiprack_1000_1["A1:H1"], use_channels=[0,1,2,3,4,5,6,7])

# await lh.discard_tips()

# await lh.aspirate(
#     binding_trough["A1"]*8,
#     vols=[120]*8,
#     use_channels=[0,1,2,3,4,5,6,7],
#     liquid_height = [8]*8,
#     flow_rates=[200]*8,
# )

In [ ]:
# # # move from mag plate to rail10[0]
# await lh.move_plate(
#     plate=binding_plate,
#     to=rail19_modules[0],   # or whatever empty holder is actually empty
#     use_arm="core",
#     pickup_distance_from_top=10,
#     channel_1=6,
#     channel_2=7,
#     core_grip_strength=60,
#     enable_recovery=False,
#     return_core_gripper=False,
# )
# await lh.backend.return_core_gripper_tools()

# await lh.move_plate(
#     plate=binding_plate,
#     to=mag_plate,   # or whatever empty holder is actually empty
#     use_arm="core",
#     pickup_distance_from_top=10,
#     channel_1=3,
#     channel_2=4,
#     core_grip_strength=60,
#     enable_recovery=False,
#     return_core_gripper=False,
# )
# await lh.backend.return_core_gripper_tools()
# lh.stop()